# Preprocess for Model Cross-Validation

In [1]:
import os
import numpy as np

In [2]:
PATH_TO_EXP = (
    "/Users/sijianfan/Documents/projects/BiSSGL/datasets/realAnalysis/cdataset"
)
PATH_DATA = os.path.join(PATH_TO_EXP, "cv_data")
if not os.path.isdir(PATH_DATA):
    os.mkdir(PATH_DATA)

In [3]:
import pandas as pd

df_Y = pd.read_table(os.path.join(PATH_TO_EXP, "c_admat_dgc.txt"), index_col=0)
print(df_Y.shape)

(409, 658)


In [4]:
df_Y.head()

,DB00014,DB00035,DB00091,DB00104,DB00115,DB00122,DB00125,DB00126,DB00131,DB00136,...,DB08801,DB08802,DB08804,DB08820,DB08824,DB08835,DB08896,DB08901,DB08906,DB08907
D102100,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
D102300,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
D102400,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
D102500,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
D103100,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Data info:

- 409 diseases $\times$ 658 drugs, binary matrix for drug resistance. 

- We save as 658 drugs $\times$ 409 diseases as $\mathbf{Y}$ to be consistant as DRIMC. 

<!-- - Not all strains are mapped with Assembly ID, so we need to remove the strains first.  -->

<!-- - The final dataset contains: 6949 strains $\times$ 13 drugs, with 9684 SNPs and 33 chemical properties.  -->

In [5]:
drugs_features = pd.read_csv(
    os.path.join(PATH_TO_EXP, "drugs_features.csv"), index_col=0
)
drugs_features.shape

(658, 1888)

In [6]:
drugs_features.head()

,ECFP_0,ECFP_1,ECFP_2,ECFP_3,ECFP_4,ECFP_5,ECFP_6,ECFP_7,ECFP_8,ECFP_9,...,GO_CC_GO:0014704,GO_CC_GO:0098552,GO_CC_GO:0031901,GO_CC_GO:0031966,GO_CC_GO:0071944,GO_CC_GO:0016600,GO_CC_GO:0045177,GO_CC_GO:0060076,GO_CC_GO:0034707,GO_CC_GO:1904813
DB00014,0,1,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DB00035,0,1,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DB00091,0,1,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
DB00104,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DB00115,1,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


Transpose Y to be consistant as DRIMC

In [7]:
Y = df_Y.T.copy()
Y.shape

(658, 409)

In [8]:
U = np.array(drugs_features, dtype=int)
U.shape

(658, 1888)

In [9]:
diseases_features = pd.read_csv(
    os.path.join(PATH_TO_EXP, "diseases_features.csv"), index_col=0
)
diseases_features.shape

(409, 797)

In [10]:
diseases_features.head()

,HPO_HP:0000006,HPO_HP:0000707,HPO_HP:0012638,HPO_HP:0012823,HPO_HP:0031797,HPO_HP:0003674,HPO_HP:0033127,HPO_HP:0001939,HPO_HP:0011446,HPO_HP:0000007,...,HPO_HP:0100273,HPO_HP:0100315,HPO_HP:0100627,HPO_HP:0100707,HPO_HP:0100738,HPO_HP:0100834,HPO_HP:0100836,HPO_HP:0100872,HPO_HP:0200008,HPO_HP:5200230
D102100,1,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
D102300,1,1,1,0,0,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
D102400,1,0,0,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
D102500,1,1,1,0,0,0,1,0,1,0,...,0,0,1,0,0,0,0,0,0,0
D103100,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
V = np.array(diseases_features, dtype=int)
V.shape

(409, 797)

In [12]:
from scipy.sparse import coo_matrix

In [13]:
U = coo_matrix(U).tocsr()
V = coo_matrix(V).tocsr()
Y = np.array(Y, dtype=int)
Y = 2 * Y - 1

In [15]:
Y

array([[-1, -1, -1, ..., -1, -1, -1],
       [-1, -1, -1, ..., -1, -1, -1],
       [-1, -1, -1, ..., -1, -1, -1],
       ...,
       [-1, -1, -1, ..., -1, -1, -1],
       [-1, -1, -1, ..., -1, -1, -1],
       [-1, -1, -1, ..., -1, -1, -1]])

In [16]:
filename_staged = os.path.join(PATH_DATA, "staged_dataset.gz")

import gzip
import pickle

with gzip.open(filename_staged, "wb+", 4) as fout:
    pickle.dump((U, V, Y), fout)